# MT-STGNN on Kaggle (2 x T4): the remaining experiments of the study
This notebook runs the Kaggle part of the study. It unpacks the code of the study (checked against the SHA-256 prefixes recorded for the Colab runs),
downloads METR-LA and verifies the file hash, builds the adjacency, runs a smoke test, and then starts a two-GPU queue
(`kaggle_run.py`, one training per GPU, the same commands as `run_plan.sh`):
**beta grid -> tuned MT-F on seeds 3-5 -> core variants on seeds 3-5 -> sensitivity runs -> 100-epoch check -> platform check**
(the last one repeats R and MT-F with beta = 1 on seeds 0-2 as a check of the machine effect). All stages together need about 9 hours on 2 x T4.

Session options needed: **Accelerator = GPU T4 x2**, **Internet = On**.

**How to run it.** Kaggle ends an interactive session after about an hour without a running cell and deletes its working directory (`/kaggle/working`),
so results that have not been downloaded are lost. For a long queue use **Save Version -> Save & Run All (Commit)**: the notebook then runs on
Kaggle's servers without that idle shutdown, cell 9 waits until the queue has finished (set `WAIT_MIN` there) and the version keeps its output
(`/kaggle/working/export`, on the Output tab of the notebook page once the version has finished). The queue skips finished runs, so cell 8 can be run again after an interruption.

**How it was run for the paper.** Three sessions of this notebook on the same kind of machine. Session 1 (interactive) ran the beta grid and was then ended by Kaggle
before the results of the later stages had been downloaded, so these stages were run again from the start in two committed versions:
version A with `STAGES = "tuned,core"` and version B with `STAGES = "xcheck,sens,e100"`. Both versions set `PRESEED` in cell 8 to the beta chosen by the grid
of session 1, so the grid was not repeated. See `COMPUTE.md` in the code package.

In [ ]:
# 1. GPUs and library versions (session options: Accelerator = GPU T4 x2, Internet = On)
import subprocess, sys
import torch
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip())
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda, "| visible GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 in the session options, then run this cell again."


In [ ]:
# 2. Unpack the code of the study and check it against the files that produced the Colab runs
import base64, hashlib, io, os, zipfile
ROOT = "/kaggle/working/mtstgnn"
PKG_SHA256 = "63bb22cddd6257bc24d06198abb1c486014244eba07f6806a944de21533a6272"
PKG_B64 = """
...
"""
raw = base64.b64decode("".join(PKG_B64.split()))
assert hashlib.sha256(raw).hexdigest() == PKG_SHA256, "the embedded package is damaged"
os.makedirs(ROOT, exist_ok=True); os.chdir(ROOT)
zipfile.ZipFile(io.BytesIO(raw)).extractall(".")
COLAB_SHA12 = {"train.py": "15c68a6fc0f9", "model.py": "bfc2c76d7de2", "data.py": "2a47f1220b2d", "metrics.py": "39a44e1339ae", "posthoc.py": "0fdc5ae2808f", "run_plan.sh": "0d5954f24953"}
got = {f: hashlib.sha256(open(f, "rb").read()).hexdigest()[:12] for f in COLAB_SHA12}
assert got == COLAB_SHA12, (got, COLAB_SHA12)
print("unpacked to", ROOT, "|", sorted(os.listdir(".")))
print("the six source files are identical to the ones of the Colab runs:", COLAB_SHA12)


In [ ]:
# 3. Dependencies (PyTorch, numpy, pandas and scipy come with the image; `tables` reads the .h5 files, `gdown` fetches the data)
import importlib, subprocess, sys
for mod in ["tables", "gdown", "scipy"]:
    try:
        importlib.import_module(mod)
    except ImportError:
        print("installing", mod)
        r = subprocess.run([sys.executable, "-m", "pip", "-q", "install", mod], capture_output=True, text=True)
        print(r.stdout[-300:], r.stderr[-300:])
import tables, scipy, gdown
print("tables", tables.__version__, "| scipy", scipy.__version__, "| gdown", gdown.__version__)


In [ ]:
# 4. METR-LA (metr-la.h5) from the Google Drive folder of the DCRNN authors, verified against the file used for the Colab runs
import glob, hashlib, os, shutil, subprocess
os.chdir("/kaggle/working/mtstgnn")
H5_SIZE, H5_SHA256 = 57038056, "64784b76d6fb8ec9bff4b6decafb354da2bb37840468fdccee5044e511277c05"
os.makedirs("/kaggle/temp/data/METR-LA", exist_ok=True); os.makedirs("/kaggle/temp/dcrnn_data", exist_ok=True)
if not os.path.exists("data"):
    os.symlink("/kaggle/temp/data", "data")          # the data stay out of the saved output
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
def find():
    for pat in ("/kaggle/input/**/metr-la.h5", "/kaggle/temp/dcrnn_data/**/metr-la.h5"):
        for p in glob.glob(pat, recursive=True):
            return p
if find() is None:
    r = subprocess.run("gdown --folder https://drive.google.com/drive/folders/10FOTa6HXPqX8Pf5WRoRwcFnW9BrNZEIX -O /kaggle/temp/dcrnn_data",
                       shell=True, capture_output=True, text=True)
    print(r.stdout[-600:], r.stderr[-600:])
src = find()
assert src, "metr-la.h5 not found: the download failed (Drive quota?). Add a METR-LA dataset as an input and run this cell again."
assert os.path.getsize(src) == H5_SIZE and sha256(src) == H5_SHA256, "metr-la.h5 is not the file used in the Colab runs"
shutil.copy(src, "data/METR-LA/metr-la.h5")
print("metr-la.h5 verified:", src, "|", H5_SIZE, "bytes | sha256", H5_SHA256[:16], "...")


In [ ]:
# 5. METR-LA adjacency (adj_mx.pkl), built from two small DATA files of the DCRNN repository (same rule and code as in the Colab notebook):
#    Gaussian kernel exp(-(d/sigma)^2), sigma = std of the road distances, entries below 0.1 set to 0
import hashlib, os, pickle, urllib.request
import numpy as np, pandas as pd
os.chdir("/kaggle/working/mtstgnn")
base = "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/"
os.makedirs("/kaggle/temp/sg", exist_ok=True)
for f in ["graph_sensor_ids.txt", "distances_la_2012.csv"]:
    urllib.request.urlretrieve(base + f, "/kaggle/temp/sg/" + f)
ids = open("/kaggle/temp/sg/graph_sensor_ids.txt").read().strip().split(",")
ind = {s: i for i, s in enumerate(ids)}
dist = np.full((len(ids), len(ids)), np.inf, dtype=np.float32)
for a, b, c in pd.read_csv("/kaggle/temp/sg/distances_la_2012.csv", dtype={"from": "str", "to": "str"}).values[:, :3]:
    if a in ind and b in ind:
        dist[ind[a], ind[b]] = float(c)
std = dist[~np.isinf(dist)].std()
adj = np.exp(-np.square(dist / std)); adj[adj < 0.1] = 0
os.makedirs("data/METR-LA", exist_ok=True)
pickle.dump([ids, ind, adj], open("data/METR-LA/adj_mx.pkl", "wb"))
nnz = int((adj > 0).sum())
same_ids = hashlib.sha256(",".join(ids).encode()).hexdigest() == "3ba026caa2e6263ab0ea54b0fa1b125dbfa7216544cd05313b555e826292b990"
same_bytes = hashlib.sha256(np.ascontiguousarray(adj, dtype=np.float32).tobytes()).hexdigest() == "e9e9099abbb2a25497c761fdf66d72114be7eb78990137514ae581f51b5ebb0f"
print(len(ids), "sensors | non-zero entries", nnz, "| sigma", float(std), "| sensor ids as in the Colab runs:", same_ids,
      "| adjacency bytes identical to the Colab one:", same_bytes)
if not same_bytes:       # float32 exp() may round differently on another CPU: then only the size of the difference matters
    a64 = np.exp(-np.square(dist.astype(np.float64) / float(std))); a64[a64 < 0.1] = 0
    print("largest |float32 - float64| entry:", float(np.abs(adj.astype(np.float64) - a64).max()))
assert len(ids) == 207 and nnz == 1722 and same_ids, "the adjacency differs from the one of the Colab runs"


In [ ]:
# 6. Dataset statistics (must equal the Colab ones) and the environment record of the runs made here
import glob, hashlib, json, os, platform, subprocess, sys, time
import numpy as np, pandas as pd, scipy, torch
os.chdir("/kaggle/working/mtstgnn"); sys.path.insert(0, ".")
from data import IGNORE, free_flow_speed, make_labels, split_ranges
os.makedirs("/kaggle/working/export", exist_ok=True)

def dataset_stats(path):
    def shares(lab):
        lab = lab[lab != IGNORE]
        return [float("nan")] * 3 if lab.size == 0 else (100.0 * np.bincount(lab, minlength=3)[:3] / lab.size).tolist()
    df = pd.read_hdf(path)
    speed = df.values.astype(np.float32); idx = df.index
    tod = ((idx.hour * 60 + idx.minute).values / 1440.0).astype(np.float32)
    T, N = speed.shape
    _, tr_end = split_ranges(T, 12, 12); va_end = int(T * 0.8)
    vf = free_flow_speed(speed, tod, tr_end); lab = make_labels(speed, vf); obs = speed > 0
    return dict(n_sensors=int(N), n_steps=int(T), start=str(idx[0]), end=str(idx[-1]), days=float(T / 288.0),
                missing_pct=float(100.0 * (~obs).mean()), ignored_label_pct=float(100.0 * (lab == IGNORE).mean()),
                level_pct_all=shares(lab), level_pct_train=shares(lab[:tr_end]), level_pct_val=shares(lab[tr_end:va_end]),
                level_pct_test=shares(lab[va_end:]), speed_mean_train=float(speed[:tr_end][obs[:tr_end]].mean()),
                speed_std_train=float(speed[:tr_end][obs[:tr_end]].std()), v_free_median=float(np.median(vf)),
                v_free_min=float(vf.min()), v_free_max=float(vf.max()),
                split_steps=dict(train=int(tr_end), val=int(va_end - tr_end), test=int(T - va_end)),
                test_days=float((T - va_end) / 288.0))

REF_DS = json.loads(r"""{"n_sensors": 207, "n_steps": 34272, "start": "2012-03-01 00:00:00", "end": "2012-06-27 23:55:00", "days": 119.0, "missing_pct": 8.109350825676486, "ignored_label_pct": 8.827363473569783, "level_pct_all": [90.28910350917987, 6.164131956641121, 3.5467645341790064], "level_pct_train": [90.6682337527889, 5.9565004685744825, 3.375265778636631], "level_pct_val": [90.54810235881726, 5.709704682546705, 3.7421929586360294], "level_pct_test": [88.74780628827618, 7.174013831003488, 4.078179880720339], "speed_mean_train": 58.58426284790039, "speed_std_train": 12.822882652282715, "v_free_median": 66.44444274902344, "v_free_min": 38.77777862548828, "v_free_max": 70.0, "split_steps": {"train": 23990, "val": 3427, "test": 6855}, "test_days": 23.802083333333332, "file": "data/METR-LA/metr-la.h5"}""")                  # dataset statistics of the Colab runs (ds_metr.json)
def close(a, b):
    if isinstance(a, dict):
        return a.keys() == b.keys() and all(close(a[k], b[k]) for k in a)
    if isinstance(a, list):
        return len(a) == len(b) and all(close(x, y) for x, y in zip(a, b))
    if isinstance(a, float):
        return abs(a - b) <= 1e-6 * max(1.0, abs(b))
    return a == b
st = dataset_stats("data/METR-LA/metr-la.h5")
bad = {k: (st[k], REF_DS[k]) for k in REF_DS if k != "file" and not close(st[k], REF_DS[k])}
assert not bad, bad
print("dataset statistics equal the Colab ones:", {k: st[k] for k in ("n_sensors", "n_steps", "start", "end", "split_steps")})
st["file"] = "data/METR-LA/metr-la.h5"
json.dump(st, open("/kaggle/working/export/ds_metr_kaggle.json", "w"), indent=1)

def cpu_model():
    for l in open("/proc/cpuinfo"):
        if l.startswith("model name"):
            return l.split(":", 1)[1].strip()
env = dict(time=time.strftime("%Y-%m-%d %H:%M:%S"), platform_name="Kaggle notebook", python=sys.version.split()[0], numpy=np.__version__,
           pandas=pd.__version__, scipy=scipy.__version__, torch=torch.__version__, cuda=torch.version.cuda,
           cudnn=torch.backends.cudnn.version(), n_gpus=torch.cuda.device_count(),
           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], gpu=torch.cuda.get_device_name(0),
           platform=platform.platform(), cpu=cpu_model(), cpu_count=os.cpu_count(),
           code_sha12={f: hashlib.sha256(open(f, "rb").read()).hexdigest()[:12]
                       for f in ["train.py", "model.py", "data.py", "metrics.py", "posthoc.py", "run_plan.sh"]})
env["nvidia_smi"] = subprocess.run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader",
                                   shell=True, capture_output=True, text=True).stdout.strip()
json.dump(env, open("/kaggle/working/export/env_kaggle.json", "w"), indent=1)
print("env:", json.dumps({k: v for k, v in env.items() if k != "code_sha12"}))


In [ ]:
# 7. Smoke test on synthetic data (about a minute): five tiny runs through the same code path as the real runs
import shutil, subprocess
r = subprocess.run("bash run_plan.sh smoke", shell=True, capture_output=True, text=True, cwd="/kaggle/working/mtstgnn")
print(r.stdout[-2200:]); print(r.stderr[-1500:])
assert r.returncode == 0, "the smoke test failed: send the whole error back"
shutil.rmtree("/kaggle/working/mtstgnn/results_smoke", ignore_errors=True)


In [ ]:
# 8. Start the two-GPU queue in the background. Safe to run again after an interruption (finished runs are skipped).
#    The stages run in the order of the list: grid -> tuned -> core (seeds 3-5) -> sens -> e100 -> xcheck (platform check on seeds 0-2).
import json, os, subprocess, time
os.chdir("/kaggle/working/mtstgnn")
STAGES = "grid,tuned,core,sens,e100,xcheck"               # all stages in one go; a shorter list for a shorter session, e.g. "grid"
PRESEED = None                                            # the beta of an earlier session's grid, e.g. {"tuned_beta": {"best": "0.5"}}; leave "grid" out of STAGES then
P = "/kaggle/working/export/status_kaggle.json"
if PRESEED and not os.path.exists(P):
    os.makedirs(os.path.dirname(P), exist_ok=True)
    json.dump({"stage": None, "running": {}, "finished": {}, "failed": [], "decisions": PRESEED}, open(P, "w"))
cmd = (f"python kaggle_run.py --stages {STAGES} --out /kaggle/working/results --export /kaggle/working/export --b1-val b1_val.json "
       "--env /kaggle/working/export/env_kaggle.json --ds /kaggle/working/export/ds_metr_kaggle.json "
       "--status /kaggle/working/export/status_kaggle.json")
if subprocess.run("pgrep -f '[k]aggle_run.py'", shell=True, capture_output=True).returncode == 0:
    print("the queue is already running")
else:
    QUEUE = subprocess.Popen(cmd, shell=True, stdout=open("/kaggle/working/queue.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
    print("queue started, pid", QUEUE.pid, "|", time.strftime("%H:%M"))


In [ ]:
# 9. Progress: a line every 5 minutes. It returns when the queue has finished or after WAIT_MIN minutes.
#    Interactive session: keep 45 and run the cell again whenever you like.
#    Save & Run All (Commit): set WAIT_MIN to more than the expected duration (at most 700; 654 was used for the paper) - the queue runs
#    in the background of this kernel and is stopped when the kernel ends, i.e. when this cell returns.
import glob, json, os, re, subprocess, time
WAIT_MIN = 45
def tail_epoch(key):
    d, name = key.split("/")
    p = f"/kaggle/working/{d}/logs/{name}.log"
    try:
        last = [l for l in open(p).read().splitlines() if l.startswith("epoch") or l.startswith("TEST") or l.startswith("early")]
        return last[-1][:70] if last else "starting"
    except Exception:
        return "?"
def snapshot():
    st = json.load(open("/kaggle/working/export/status_kaggle.json"))
    out = [f"{time.strftime('%H:%M')} | stage {st['stage']} | runs finished {len(st['finished'])} | failed {st['failed']}"]
    for k, v in sorted(st["running"].items(), key=lambda kv: kv[1]["gpu"]):
        out.append(f"    GPU {v['gpu']}: {k} (since {v['start']}) {tail_epoch(k)}")
    g = subprocess.run("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
    out.append("    nvidia-smi: " + g.replace("\n", " | "))
    if st["decisions"]:
        out.append("    decisions: " + json.dumps(st["decisions"])[:300])
    return "\n".join(out)
t0 = time.time()
while True:
    try:
        s = snapshot(); print(s, flush=True)
        if json.load(open("/kaggle/working/export/status_kaggle.json"))["stage"] == "ALL DONE":
            print("ALL DONE"); break
    except Exception as e:
        print("no status yet:", repr(e), flush=True)
    if time.time() - t0 > WAIT_MIN * 60:
        break
    time.sleep(300)


In [ ]:
# 10. Files to take away (the thin result zips are the only things needed): /kaggle/working/export
#     (Output panel of an interactive session; Output tab of the notebook page for a committed version after it has finished)
import glob, hashlib, os
for p in sorted(glob.glob("/kaggle/working/export/*")):
    print(f"{os.path.getsize(p):>10,}  {hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16]}  {p}")
print(len(glob.glob("/kaggle/working/results/*_s?.json")), "run files in /kaggle/working/results |",
      len(glob.glob("/kaggle/working/results_e100/*_s?.json")), "in results_e100 |",
      len(glob.glob("/kaggle/working/results_xcheck/*_s?.json")), "in results_xcheck")
